Chat Parser

In [2]:
# Feature 1 - Chat Parser

from datetime import datetime

# Read the WhatsApp chat file
with open("hostel_bois.txt", "r", encoding="utf-8") as file:
    lines = file.readlines()

messages = []

system_messages = 0
media_messages = 0
deleted_messages = 0

for line in lines:

    line = line.strip()

    # Skip empty lines
    if not line:
        continue

    # Check whether the line starts with a date
    try:
        timestamp, rest = line.split(" - ", 1)

        dt = datetime.strptime(timestamp, "%d/%m/%y, %H:%M")

    except:
        # Multi-line continuation
        continue

    # System message: no ": " after sender
    if ": " not in rest:
        system_messages += 1
        continue

    # Separate sender and message
    sender, message = rest.split(": ", 1)

    # Count media messages
    if message == "<Media omitted>":
        media_messages += 1

    # Count deleted messages
    if message == "This message was deleted":
        deleted_messages += 1

    # Store the message
    messages.append({
        "datetime": dt,
        "date": dt.date(),
        "time": dt.time(),
        "hour": dt.hour,
        "sender": sender,
        "message": message
    })


# Output
print("CHAT PARSER")
print("=" * 50)

print("Successfully parsed:", len(messages), "messages")
print("System messages:", system_messages)
print("Media messages:", media_messages)
print("Deleted messages:", deleted_messages)

print("\nFirst 5 messages:")

for msg in messages[:5]:
    print(
        msg["datetime"].strftime("%d/%m/%y, %H:%M"),
        "-",
        msg["sender"] + ":",
        msg["message"]
    )

CHAT PARSER
Successfully parsed: 3174 messages
System messages: 4
Media messages: 32
Deleted messages: 15

First 5 messages:
01/04/24, 01:17 - Rahul: scene fix
01/04/24, 01:17 - Rahul: haan
01/04/24, 01:18 - Rahul: kya scene
01/04/24, 02:13 - Rahul: abhi free hai?
01/04/24, 02:13 - Rahul: abey


Group Overview

In [5]:

# FEATURE 2 - GROUP OVERVIEW

from collections import Counter

# Total messages
total_messages = len(messages)

# Participants
participants = sorted(set(msg["sender"] for msg in messages))

# Date range
dates = [msg["datetime"] for msg in messages]
first_date = min(dates)
last_date = max(dates)

# Number of days
total_days = (last_date.date() - first_date.date()).days + 1

# Messages per person
message_counts = Counter(msg["sender"] for msg in messages)

# Percentage per person
print("=" * 60)
print("                 GROUP OVERVIEW")
print("=" * 60)

print(f"Group          : Hostel Bois 4ever")
print(f"Period         : {first_date.strftime('%d %B %Y')} to {last_date.strftime('%d %B %Y')} ({total_days} days)")
print(f"Total messages : {total_messages}")
print(f"Participants   : {len(participants)}")

print("\nMESSAGES PER PERSON")
print("-" * 40)

for person, count in message_counts.most_common():
    percentage = (count / total_messages) * 100
    print(f"{person:<12} : {count:>4} ({percentage:.1f}%)")

print("=" * 60)

                 GROUP OVERVIEW
Group          : Hostel Bois 4ever
Period         : 01 April 2024 to 30 May 2024 (60 days)
Total messages : 3174
Participants   : 6

MESSAGES PER PERSON
----------------------------------------
Rahul        :  953 (30.0%)
Priya        :  718 (22.6%)
Neha         :  635 (20.0%)
Aman         :  490 (15.4%)
Karan        :  354 (11.2%)
Vikas        :   24 (0.8%)


Most Active Day & Hour

In [4]:

# FEATURE 3 - MOST ACTIVE DAY & HOUR

from collections import Counter

# Count messages for each date
day_counts = Counter(msg["datetime"].date() for msg in messages)

# Count messages for each hour
hour_counts = Counter(msg["datetime"].hour for msg in messages)

# Find busiest day and busiest hour
busiest_day, busiest_day_count = day_counts.most_common(1)[0]
busiest_hour, busiest_hour_count = hour_counts.most_common(1)[0]

# Average messages per day
average_per_day = len(messages) / len(day_counts)

print("=" * 60)
print("              MOST ACTIVE DAY & HOUR")
print("=" * 60)

print(
    f"Busiest day  : {busiest_day.strftime('%d %B %Y')} "
    f"({busiest_day_count} messages)"
)

print(
    f"Busiest hour : {busiest_hour:02d}:00 - {busiest_hour:02d}:59 "
    f"({hour_counts[busiest_hour]} messages)"
)

print(
    f"Average      : {average_per_day:.0f} messages per day"
)

print("=" * 60)

              MOST ACTIVE DAY & HOUR
Busiest day  : 04 May 2024 (76 messages)
Busiest hour : 18:00 - 18:59 (248 messages)
Average      : 53 messages per day


Numpy Activity Heatmap

In [6]:

# FEATURE 4 - NUMPY ACTIVITY HEATMAP


import numpy as np

# Get participants from the parsed messages
participants = sorted(set(msg["sender"] for msg in messages))

# Create matrix: participants × 24 hours
activity = np.zeros((len(participants), 24), dtype=int)

# Count messages by person and hour
for msg in messages:
    person = msg["sender"]
    hour = msg["datetime"].hour

    person_index = participants.index(person)
    activity[person_index, hour] += 1

print("=" * 80)
print("                 ACTIVITY HEATMAP")
print("=" * 80)

# Print hour headings
print(f"{'Person':<15}", end="")

for hour in range(24):
    print(f"{hour:>4}", end="")

print()

# Print activity matrix
for i, person in enumerate(participants):
    print(f"{person:<15}", end="")

    for hour in range(24):
        print(f"{activity[i, hour]:>4}", end="")

    print()

print("\nMatrix shape:", activity.shape)
print("Total messages in matrix:", activity.sum())

                 ACTIVITY HEATMAP
Person            0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18  19  20  21  22  23
Aman             54  67  66  60  88   0   0   0   0   0   0   0   0   0  14  11  19   7  16   8  13  11   0  56
Karan             0   0   0   0   0   0   0   4  12  16  20  16  37  25  32  27  27  27  25  32  23  14   9   8
Neha              0   0   0   0   0  19   3  13  36  52  52  22  39  36  27  10  37  47  62  50  45  27  28  30
Priya             0   0   0   0   0   0  13  20  47  65  62  61  57  48  44  29  32  40  38  60  43  32  18   9
Rahul             3  15  17  17  22  10  17  17  24  17  25  15  58  48  45  53  73  49 105  76  41  92  60  54
Vikas             0   0   0   0   0   0   0   1   3   1   1   0   2   2   0   1   1   3   2   2   1   1   1   2

Matrix shape: (6, 24)
Total messages in matrix: 3174


Response Speed & Silent Streaks

In [7]:
# Feature 5 - Response Speed & Silent Streaks

print("RESPONSE SPEED & SILENT STREAKS")
print("=" * 50)

# Response time calculation
response_times = []

for i in range(1, len(messages)):
    previous = messages[i - 1]
    current = messages[i]

    # Only calculate when sender changes
    if previous["sender"] != current["sender"]:
        difference = current["datetime"] - previous["datetime"]
        minutes = difference.total_seconds() / 60

        # Ignore very large gaps
        if minutes <= 60:
            response_times.append(minutes)

if len(response_times) > 0:
    average_response = sum(response_times) / len(response_times)
else:
    average_response = 0

print("Total responses analyzed:", len(response_times))
print("Average response time:", round(average_response, 2), "minutes")


# Silent streaks
silent_streaks = []

for i in range(1, len(messages)):
    previous = messages[i - 1]
    current = messages[i]

    difference = current["datetime"] - previous["datetime"]
    hours = difference.total_seconds() / 3600

    if hours >= 6:
        silent_streaks.append(hours)

print("Silent streaks (6+ hours):", len(silent_streaks))

if len(silent_streaks) > 0:
    longest_silent = max(silent_streaks)
    print("Longest silent streak:", round(longest_silent, 2), "hours")
else:
    print("Longest silent streak: No long gap found")

RESPONSE SPEED & SILENT STREAKS
Total responses analyzed: 1076
Average response time: 20.62 minutes
Silent streaks (6+ hours): 0
Longest silent streak: No long gap found


Personality Archtypes

In [8]:
# Feature 6 - Personality Archetypes

print("PERSONALITY ARCHETYPES")
print("=" * 50)

# Count messages and words for each participant
message_count = {}
word_count = {}

for msg in messages:
    sender = msg["sender"]

    if sender not in message_count:
        message_count[sender] = 0
        word_count[sender] = 0

    message_count[sender] += 1
    word_count[sender] += len(msg["message"].split())

# Find most active participant
most_active = max(message_count, key=message_count.get)

# Assign simple archetypes
for person in participants:
    messages_sent = message_count.get(person, 0)
    words_sent = word_count.get(person, 0)

    if person == most_active:
        archetype = "The Leader"
    elif words_sent > 1000:
        archetype = "The Talker"
    elif messages_sent < 50:
        archetype = "The Silent Observer"
    elif words_sent / max(messages_sent, 1) < 3:
        archetype = "The Short Responder"
    else:
        archetype = "The Regular"

    print("\nParticipant:", person)
    print("Messages:", messages_sent)
    print("Words:", words_sent)
    print("Archetype:", archetype)

PERSONALITY ARCHETYPES

Participant: Aman
Messages: 490
Words: 2446
Archetype: The Talker

Participant: Karan
Messages: 354
Words: 19703
Archetype: The Talker

Participant: Neha
Messages: 635
Words: 3345
Archetype: The Talker

Participant: Priya
Messages: 718
Words: 3576
Archetype: The Talker

Participant: Rahul
Messages: 953
Words: 2437
Archetype: The Leader

Participant: Vikas
Messages: 24
Words: 44
Archetype: The Silent Observer


Project Report

In [9]:
# Feature 7 - Final Report

print("\n")
print("=" * 60)
print("              GROUPDNA - FINAL REPORT")
print("=" * 60)

# Basic statistics
total_messages = len(messages)
total_participants = len(participants)

# Message counts
message_count = {}

for msg in messages:
    sender = msg["sender"]

    if sender in message_count:
        message_count[sender] += 1
    else:
        message_count[sender] = 1

# Most active participant
most_active = max(message_count, key=message_count.get)

# Date range
first_date = min(msg["datetime"] for msg in messages)
last_date = max(msg["datetime"] for msg in messages)

print("\nGROUP OVERVIEW")
print("-" * 40)
print("Total Messages       :", total_messages)
print("Total Participants   :", total_participants)
print("First Message        :", first_date.strftime("%d-%m-%Y %H:%M"))
print("Last Message         :", last_date.strftime("%d-%m-%Y %H:%M"))
print("Most Active Member   :", most_active)
print("Messages by Most Active:", message_count[most_active])

print("\nPARTICIPANT ACTIVITY")
print("-" * 40)

for person in participants:
    print(person, ":", message_count.get(person, 0), "messages")

print("\nRESPONSE ANALYSIS")
print("-" * 40)
print("Responses Analyzed   :", len(response_times))
print("Average Response     :", round(average_response, 2), "minutes")
print("Silent Streaks       :", len(silent_streaks))

print("\nNUMPY ACTIVITY MATRIX")
print("-" * 40)
print("Matrix Shape         :", activity.shape)
print("Matrix Total         :", activity.sum())

print("\n")
print("=" * 60)
print("             GROUPDNA ANALYSIS COMPLETE")
print("=" * 60)



              GROUPDNA - FINAL REPORT

GROUP OVERVIEW
----------------------------------------
Total Messages       : 3174
Total Participants   : 6
First Message        : 01-04-2024 01:17
Last Message         : 30-05-2024 23:31
Most Active Member   : Rahul
Messages by Most Active: 953

PARTICIPANT ACTIVITY
----------------------------------------
Aman : 490 messages
Karan : 354 messages
Neha : 635 messages
Priya : 718 messages
Rahul : 953 messages
Vikas : 24 messages

RESPONSE ANALYSIS
----------------------------------------
Responses Analyzed   : 1076
Average Response     : 20.62 minutes
Silent Streaks       : 0

NUMPY ACTIVITY MATRIX
----------------------------------------
Matrix Shape         : (6, 24)
Matrix Total         : 3174


             GROUPDNA ANALYSIS COMPLETE
